## Открытие демонстрационных текстовых данных.

> Стоит отметить что несмотря на то что у нас сырой текст мы строго соблюдаем синтаксис markdown,т.к таким образом легче ориентироваться по всему документу.

In [86]:
import numpy as np
import re
import collections

f = open('../data/raw/sp_120.13330.txt', encoding='utf-8')
print(f.readlines())

['## 3. Термины, определения и сокращения\n', '### 3.1 Термины и определения\n', '\n', 'В настоящем своде правил применены следующие термины с соответствующими определениями:\n', '\n', '3.1.1 автоматизированное рабочее место диспетчера; АРМ: Комплекс технических средств, позволяющих диспетчеру соответствующих подразделений метрополитена управлять оборудованием и получать достоверную информацию о его техническом состоянии в любое время.\n', '\n', '3.1.2 акт незаконного вмешательства: Противоправное действие (бездействие), в том числе террористический акт, угрожающее безопасной деятельности транспортного комплекса, повлекшее за собой причинение вреда жизни и здоровью людей, материальный ущерб либо создавшее угрозу наступления таких последствий.\n', '\n', '3.1.3\n', '\n', 'аппарель: Элемент обустройства пешеходного пути в виде монолитной или накладной конструкции, в том числе на лестничном марше или через препятствие, состоящий из двух раздельных направляющих для перемещения средств на ко

In [87]:
f.readlines()
# Как видим если не закрывать файл
# то указатель постоянно будет в (EOF) конце файла,
# что кстати неудобно при работе в jupyter.
# Например если надо посчитать некоторый встречающийся кусок текста после того как уже прочитали файл 

[]

In [88]:
f.close()

## Анализ структуры
### Количество заголовок подскажет, по каким маркерам делать chunking (например, разбивать по хэдерам - `###`).

In [89]:
with open("../data/raw/sp_120.13330.txt", "r", encoding="utf-8") as f:
    text = f.read()  

h1 = text.count('# ')
h2 = text.count('## ')
h3 = text.count('### ')
h4 = text.count('#### ')

print(f'Количество заголовок:\n H1 - {h1},\n H2 - {h2},\n H3 - {h3},\n H4 - {4}')

Количество заголовок:
 H1 - 4,
 H2 - 4,
 H3 - 2,
 H4 - 4


## Статистика по длине
### Помогает выбрать размер чанка (например, 1 пункт = 1 чанк).

In [90]:
lines = text.split('\n')
paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

print(f"Всего строк: {len(lines)}")
print(f"Абзацев/пунктов: {len(paragraphs)}")
print(f"Средняя длина пункта: {np.mean([len(p) for p in paragraphs]):.0f} символов")

Всего строк: 558
Абзацев/пунктов: 278
Средняя длина пункта: 117 символов


## Наличие таблиц
### Если таблицы есть — значит, нужно сохранять их как единый блок при chunking.

In [91]:
has_tables = "|---" in text or "Согласно таблице" in text.lower()
print("Таблицы обнаружены:", has_tables)

# Пример: извлечь все строки, похожие на таблицу
table_lines = [line for line in lines if line.count("|") >= 2]
print(f"Возможные строки таблиц: {len(table_lines)}")

Таблицы обнаружены: False
Возможные строки таблиц: 0


## Анализ терминологии
### Показывает, насколько отрывок релевантен теме.

In [93]:
words = re.findall(r'\b\w+\b', text.lower())
freq = collections.Counter(words)

# Доменные термины для метрополитена. Нужно дополнить!!!
domain_terms = ['станция', 'платформа', 'тоннель', 'эскалатор', 'обделка', 'уклон', 'ширина', 'требование']

print("Ключевые термины в отрывке:")
for term in domain_terms:
    if term in freq:
        print(f"  {term}: {freq[term]}")

Ключевые термины в отрывке:
  станция: 4
  тоннель: 6
  обделка: 1


## Проверка на шум
### Есть ли артефакты копирования?
`("●", "", "стр. 45")`

In [94]:
noise_patterns = ["изменения", "доступен профессиональным", "редакция от", "страница"]
noise_found = [p for p in noise_patterns if p in text.lower()]
print("Обнаружен шум:", noise_found) # Нужно дополнить!!!

Обнаружен шум: []
